# 07 · Spatiotemporal Dataset Construction

Combines weather + spatial + temporal + lagged features into an ML-ready dataset.

> **Prerequisite:** `06_weather_extraction` (need `weather_data_raw.csv`)

In [ ]:
import sys
if '/content/geo_sp_project/src' not in sys.path:
    sys.path.extend(['/content/geo_sp_project/src', '/content/geo_sp_project/configs'])


In [ ]:
import pandas as pd, config
from geo_sp.spatiotemporal import (
    to_geodataframe, add_buffers, add_temporal_features,
    add_derived_weather, add_spatial_features, add_lagged_features,
    save_dataset, make_feature_docs, save_metadata, validate
)

# Load weather data
try:
    df = pd.read_parquet(config.WEATHER_FINAL_FILE.replace('.csv', '.parquet'))
except Exception:
    df = pd.read_csv(config.WEATHER_FINAL_FILE)

df['date'] = pd.to_datetime(df['date'])
df_sites   = pd.read_csv(config.SITE_REGISTRY_CSV)
print(f'Records: {len(df):,}  |  Sites: {df["site_id"].nunique()}')


In [ ]:
BUFFER_SIZES = {'buffer_500m': 500, 'buffer_1km': 1000, 'buffer_5km': 5000}

gdf = to_geodataframe(df)
gdf = add_buffers(gdf, BUFFER_SIZES)
gdf = add_temporal_features(gdf)
gdf = add_derived_weather(gdf)
gdf = add_spatial_features(gdf)
gdf = add_lagged_features(gdf)

print(f'\nFinal dataset: {len(gdf):,} rows × {len(gdf.columns)} columns')


In [ ]:
save_dataset(gdf, list(BUFFER_SIZES.keys()),
             'spatiotemporal_dataset.csv',
             'spatiotemporal_dataset.parquet',
             'spatiotemporal_dataset.gpkg')

docs = make_feature_docs(gdf, ['geometry'] + list(BUFFER_SIZES.keys()))
docs.to_csv('feature_documentation.csv', index=False)
save_metadata(gdf, BUFFER_SIZES, 'dataset_metadata.json')
print('✅ All files saved')


In [ ]:
v = validate(gdf, df_sites)
for k, result in v.items():
    print(f'  {k}: {"✅ PASS" if result else "❌ FAIL"}')
